In [ ]:
from pathlib import Path
import os
import sys

def _find_kpihub_root() -> Path:
    starters = [Path.cwd().resolve()]
    nb_file = globals().get("__vsc_ipynb_file__") or globals().get("__file__")
    if nb_file:
        starters.insert(0, Path(nb_file).resolve().parent)

    seen = set()
    for start in starters:
        for p in [start, *list(start.parents)[:12]]:
            if p in seen:
                continue
            seen.add(p)
            if (p / "lib" / "tables").is_dir():
                return p
            kpihub = p / "KPIHub"
            if (kpihub / "lib" / "tables").is_dir():
                return kpihub

    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )


_root = _find_kpihub_root()
sys.path.insert(0, str(_root))
os.chdir(_root)

notebook_dir = Path().resolve()        

In [6]:

import pandas as pd

from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.CustomerHandler import *


# Init the SAT customers

In [7]:
customer_list = pd.read_csv(os.path.join(notebook_dir, 'init', 'PeaksAboveSAT_Customer.csv'))
customer_ids = []
for _, customer in customer_list.iterrows():
    result = Query(query=f"SELECT C.Id as CustomerId, C.Name FROM Customer C WHERE LOWER(C.Name) = LOWER('{customer['Name']}')").execute(CONN_DICT[customer['DBLocation']])
    if not result.empty:
        customer_ids.append(result.iloc[0]['CustomerId'])
    else:
        customer_ids.append(None)
customer_list['CustomerId'] = customer_ids
customer_list['LastUpdated'] = pd.Timestamp.now()
PeakAboveSATCustomer.update_table(arguments={'db_path': DB_PATH, 'DataFrame': customer_list, 'PrimaryKey': 'CustomerId'})
PeakAboveSATCustomer.query_table(arguments={'db_path': DB_PATH})


,CustomerId,Name,ShortName,Active,DBLocation,XchangeLocation,BoxFolderId,ThresholdSCFH,LastUpdated
0,BD4D080B-1D12-D329-ABD0-39FEB9804E98,Cadent,None,1,EU2,v_boundary_cadent_2026_4326,370307867916,21.15,2026-09-07 08:38:27.242270
1,027114F8-DDB7-C0D0-1398-3A173A08C9BE,Wales and West Utilities,None,0,EU1,v_boundary_wwu_2025_4326,414908775896,21.15,2026-09-07 08:38:27.242270
2,A92B3A0B-7FEB-D28B-4000-3A1C7491E542,SGN,None,0,EU1,v_boundary_sgn_2026_4326,414909630436,21.15,2026-09-07 08:38:27.242270
3,77979D80-4767-B9A3-5E89-3A182BB7A17A,Northern Gas Networks,None,0,EU1,v_boundary_ngn_2026_4326,414920005301,21.15,2026-09-07 08:38:27.242270


# Init the mailing list of the peak above SAT

In [8]:
mailing_list = pd.read_csv(os.path.join(notebook_dir, 'init', 'PeaksAboveSAT_MailList.csv'))
mailing_list = pd.merge(mailing_list, customer_list[['Name', 'CustomerId']], left_on='Customer', right_on='Name', how='left')
mailing_list.drop(columns = ['Name', 'Customer'], inplace = True)
mailing_list['LastUpdated'] = pd.Timestamp.now()
PeakAboveSATRecipients.update_table(arguments={'db_path': DB_PATH, 'DataFrame': mailing_list, 'PrimaryKey': ['CustomerId', 'Email']})
PeakAboveSATRecipients.query_table(arguments={'db_path': DB_PATH})


,CustomerId,Email,Active,LastUpdated
0,BD4D080B-1D12-D329-ABD0-39FEB9804E98,dsoler@picarro.com,1,2026-09-07 08:38:27.286062
1,BD4D080B-1D12-D329-ABD0-39FEB9804E98,distrate@picarro.com,0,2026-09-07 08:38:27.286062
2,BD4D080B-1D12-D329-ABD0-39FEB9804E98,tberhanu@picarro.com,0,2026-09-07 08:38:27.286062
3,BD4D080B-1D12-D329-ABD0-39FEB9804E98,theo@veho-solutions.co.uk,0,2026-09-07 08:38:27.286062
4,BD4D080B-1D12-D329-ABD0-39FEB9804E98,sian@veho-solutions.co.uk,0,2026-09-07 08:38:27.286062
5,BD4D080B-1D12-D329-ABD0-39FEB9804E98,box.ALD@cadentgas.com,0,2026-09-07 08:38:27.286062
6,BD4D080B-1D12-D329-ABD0-39FEB9804E98,firas@veho-solutions.co.uk,0,2026-09-07 08:38:27.286062
7,027114F8-DDB7-C0D0-1398-3A173A08C9BE,dsoler@picarro.com,1,2026-09-07 08:38:27.286062
8,027114F8-DDB7-C0D0-1398-3A173A08C9BE,tberhanu@picarro.com,0,2026-09-07 08:38:27.286062
9,027114F8-DDB7-C0D0-1398-3A173A08C9BE,bschmitt@picarro.com,0,2026-09-07 08:38:27.286062
